# Week 3 — Speed and subnetwork extraction

**Goal:** make the brain fast enough for the closed loop (≤ 10 s of compute per 1 s of brain time).

1. **Verify** the new simulators against Brian2 on the full network with *identical input spikes*: the float64 engines should reproduce Brian2's spikes exactly.
2. **Reference runs** of the full network (JO-C/E stimulus, chosen `w_syn`).
3. **Subnetworks**: path-length subnetworks and an activity-based one, each scored against the full network with a pass rule fixed in advance.
4. **Benchmark**: seconds of compute per simulated second, full vs subnetwork, CPU vs GPU.
5. **Decision.**

**Where to run:** Colab with **Runtime → Change runtime type → T4 GPU**. Then (optional, recommended) run it again on your laptop: GPU rows are skipped there, and results are saved per machine, so Step 5 can decide for both. Total time on Colab ≈ 15–25 min.

### Setup

In [1]:
import sys, subprocess
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    try:
        import brian2
    except ImportError:
        # installing brian2 can upgrade numpy, which breaks the numpy already loaded:
        # install, then restart the runtime. Afterwards, run all cells again.
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'brian2', 'numba'], check=True)
        print("brian2 installed -> restarting runtime. When it reconnects, choose Runtime -> Run all again.")
        import os; os.kill(os.getpid(), 9)
except ImportError:
    IN_COLAB = False

from pathlib import Path
BRAIN = Path('/content/drive/MyDrive/arm_simulation/brain') if IN_COLAB else Path.home() / 'arm_simulation/brain'
DATA = BRAIN / 'data'
sys.path.insert(0, str(BRAIN))
for p in ['sim/lif.py', 'sim/lif_torch.py', 'sim/lif_numba.py', 'sim/bench.py',
          'data/weights_male-cns_v1.0/meta.json', 'data/shortlist_week1.csv', 'data/wsyn_choice.json']:
    print(f"{p:45s} {'ok' if (BRAIN / p).exists() else 'MISSING'}")

sim/lif.py                                    ok
sim/lif_torch.py                              ok
sim/lif_numba.py                              ok
sim/bench.py                                  ok
data/weights_male-cns_v1.0/meta.json          ok
data/shortlist_week1.csv                      ok
data/wsyn_choice.json                         ok


### Load weights, groups and settings

In [2]:
import json, time, gc, platform, os
import numpy as np, pandas as pd, torch
import warnings; warnings.filterwarnings('ignore', module='brian2')
from sim.lif import WeightSet, peak_mem_gb
from sim.bench import (B2Fixed, poisson_inputs, spike_match, readout, rate_agreement, path_subnets,
                       active_subnet, make_engine, time_engine, gpu_available, ENGINES)

ws = WeightSet(DATA / 'weights_male-cns_v1.0')
W = json.loads((DATA / 'wsyn_choice.json').read_text())['w_scale']
JO = ws.by_type(r'^JO-(C|E)', cls='cb_sensory')
shortlist = pd.read_csv(DATA / 'shortlist_week1.csv')
g2i = pd.Series(ws.neurons.index, index=ws.neurons.bodyId)
DN_cand = g2i.loc[shortlist.query("group == 'middle_descending'").bodyId].to_numpy()
LEG_MN = g2i.loc[shortlist.query("group == 'output_T1_leg_MN'").bodyId].to_numpy()
READ = {'DN': DN_cand, 'legMN': LEG_MN}
RATE = 150                    # Hz, JO-C/E stimulus, as in Week 2

GPU = gpu_available()
HOST = 'colab' if IN_COLAB else 'laptop'
torch.set_num_threads(os.cpu_count())
OUT = DATA / 'week3'; OUT.mkdir(exist_ok=True)
# engines: the 'reference' engine runs the many trials of Steps 2-3 (float64, exact)
REF_ENGINE = 'torch-gpu-f64' if GPU else 'numba'
gpu_name = torch.cuda.get_device_name(0) if GPU else 'none'
print(f"host {HOST}: {os.cpu_count()} CPU cores, GPU {gpu_name}, reference engine {REF_ENGINE}")
print(f"w_scale {W}, JO-C/E {len(JO)}, DN candidates {len(DN_cand)}, leg MNs {len(LEG_MN)}, "
      f"peak memory {peak_mem_gb():.2f} GB")

host laptop: 8 CPU cores, GPU none, reference engine numba
w_scale 0.7, JO-C/E 335, DN candidates 68, leg MNs 81, peak memory 1.12 GB


## Step 1: verify against Brian2 (same input spikes)

Brian2 gets its JO-C/E input from a fixed spike array instead of a PoissonGroup, and every new engine gets the same array. Pass rule: the **float64 engines must match Brian2 spike for spike** (same neuron, same 0.1 ms step); float32 is allowed to drift (a single rounding difference can flip one spike, and the network amplifies it), so it is judged by rates.

On Colab this runs the **full network** (Brian2 needs ~2.5 GB here). On the laptop (4 GB) it uses the JO subnetwork from Week 2 (path length ≤ 4) instead.

One model detail found while porting: because `v` and `g` are declared `(unless refractory)`, **Brian2 discards synaptic input that arrives while the target neuron is refractory** (including on the step it spikes). The new engines do the same; without it, ~5% of spikes differed.

In [3]:
full = ws.subnetwork(np.arange(ws.N))
if IN_COLAB:
    VER_NET, ver_sub = 'full', full
else:
    VER_NET, ver_sub = 'path L=4', path_subnets(ws, JO, np.r_[DN_cand, LEG_MN], [4], verbose=False)['path L=4']
inp_ver = poisson_inputs(len(JO), RATE, 1.0, n_trials=2, seed=7)

b2 = B2Fixed(ver_sub, JO)
s_b2 = b2.run(inp_ver, w_scale=W)
B2_TIME = {VER_NET: dict(build_s=b2.build_s, s_per_sim_s=b2.run_s[-1])}   # 2nd trial: no compile time
keep_b2 = b2.keep
r_b2 = b2.rates(s_b2)
del b2; gc.collect()

rows = [dict(engine='brian2', **readout(r_b2, READ), spikes=len(s_b2), match_frac=1.0, first_mismatch_s=None)]
for kind, (desc, needs_gpu) in ENGINES.items():
    if needs_gpu and not GPU:
        continue
    eng = make_engine(kind, ver_sub, JO)
    s = eng.run(input_spikes=inp_ver, w_scale=W, verbose=False)
    m = spike_match(s_b2, keep_b2, s, eng.keep)
    rows.append(dict(engine=kind, **readout(eng.rates(s), READ), spikes=len(s),
                     match_frac=round(m['match_frac'], 5), first_mismatch_s=m['first_mismatch_s'],
                     rate_corr=rate_agreement(r_b2, eng.rates(s), keep_b2)['corr']))
    print(f"{kind:15s} match {m['match_frac']:.5f}  first mismatch {m['first_mismatch_s']}")
    del eng; gc.collect()
verify = pd.DataFrame(rows).assign(network=VER_NET, host=HOST)
verify.to_csv(OUT / f'verify_{HOST}.csv', index=False)
exact = verify[verify.engine.isin(['numba', 'torch-gpu-f64'])]
print('\nfloat64 engines exact:', bool((exact.match_frac >= 0.999).all()))
display(verify.round(4))

Brian2: 18,648 neurons, 3,039,649 connections, built in 23.1 s
Brian2: 2 trials, 21.7 s per trial
numba           match 1.00000  first mismatch None
torch-cpu       match 0.28019  first mismatch 0.0284

float64 engines exact: True


,engine,DN_firing,DN_mean_hz,legMN_firing,legMN_mean_hz,spikes,match_frac,first_mismatch_s,rate_corr,network,host
0,brian2,39,42.3603,9,5.5247,425623,1.0000,NaN,NaN,path L=4,laptop
1,numba,39,42.3603,9,5.5247,425623,1.0000,NaN,1.0000,path L=4,laptop
2,torch-cpu,39,42.3015,9,5.5185,425113,0.2802,0.0284,0.9998,path L=4,laptop


## Step 2: full-network reference runs

Two **test** conditions, 10 trials each, with fixed input spikes so every subnetwork sees exactly the same inputs as the full network:
- A: JO-C/E at 150 Hz (the Week 2 condition)
- B: JO-C/E at 75 Hz (a weaker stimulus, as the closed loop will vary the input)

Plus a separate **build** set (75 / 150 / 300 Hz, 5 trials each, different seeds) used only to define the activity-based subnetwork, so it is scored on runs it has not seen.

In [4]:
TEST = {'A 150Hz': poisson_inputs(len(JO), 150, 1.0, 10, seed=100),
        'B 75Hz':  poisson_inputs(len(JO), 75, 1.0, 10, seed=200)}
BUILD = {hz: poisson_inputs(len(JO), hz, 1.0, 5, seed=1000 + hz) for hz in (75, 150, 300)}

t0 = time.perf_counter()
ref = make_engine(REF_ENGINE, full, JO)
full_spk = {c: ref.run(input_spikes=inp, w_scale=W, verbose=False) for c, inp in TEST.items()}
full_rates = {c: ref.rates(s) for c, s in full_spk.items()}
build_spk = [(ref.run(input_spikes=inp, w_scale=W, verbose=False), ref.keep) for inp in BUILD.values()]
full_keep = ref.keep
del ref; gc.collect()
for c, r in full_rates.items():
    others = r.drop(JO)
    print(f"{c}: {readout(r, READ)}, network firing {(others >= 1).mean():.2%}")
print(f"({time.perf_counter() - t0:.0f} s)")

A 150Hz: {'DN_firing': 39, 'DN_mean_hz': 41.49558823529412, 'legMN_firing': 9, 'legMN_mean_hz': 5.622222222222223}, network firing 4.78%
B 75Hz: {'DN_firing': 37, 'DN_mean_hz': 22.46764705882353, 'legMN_firing': 10, 'legMN_mean_hz': 5.591358024691359}, network firing 3.16%
(196 s)


## Step 3: subnetworks

**Path subnetworks** (as in Week 2): neurons on a JO-C/E → (DN candidates ∪ leg MNs) path of ≤ L hops, counting only connections with ≥ 5 synapses; all connections among the kept neurons are simulated.

**Active set**: every neuron that fired at least once in the build runs, plus the readout neurons. A neuron that never spikes has no effect on any other neuron, so this is exact on those runs. The question is how well it generalizes to new inputs, which the test set measures.

**Pass rule (fixed before seeing results)**, in both test conditions against the full network:
- DN candidates: per-neuron rate correlation ≥ 0.95 and mean rate within ±10%
- leg MNs: number firing (≥ 1 Hz) within ±2

In [5]:
DST = np.r_[DN_cand, LEG_MN]
subs = path_subnets(ws, JO, DST, [3, 4, 5, 6])
subs['active set'] = active_subnet(ws, build_spk, extra=DST)
subs = {k: s for k, s in subs.items() if len(s['w']) < 0.8 * len(full['w'])}   # skip near-full ones

def fidelity(name, sub):
    eng = make_engine(REF_ENGINE, sub, JO)
    row = dict(network=name, neurons=len(sub['keep']), connections=len(sub['w']))
    ok = True
    for c, inp in TEST.items():
        s = eng.run(input_spikes=inp, w_scale=W, verbose=False)
        r = eng.rates(s)
        dn = rate_agreement(full_rates[c], r, DN_cand)
        ro_full, ro = readout(full_rates[c], READ), readout(r, READ)
        sm = spike_match(full_spk[c], full_keep, s, eng.keep, neurons=DST)
        row.update({f'{c}: DN corr': dn['corr'], f'{c}: DN mean ratio': dn['mean_ratio'],
                    f'{c}: legMN firing': f"{ro['legMN_firing']} (full {ro_full['legMN_firing']})",
                    f'{c}: readout spikes exact': sm['match_frac']})
        ok &= (dn['corr'] >= 0.95) and abs(dn['mean_ratio'] - 1) <= 0.10 \
              and abs(ro['legMN_firing'] - ro_full['legMN_firing']) <= 2
    row['pass'] = bool(ok)
    del eng; gc.collect()
    return row

fid = pd.DataFrame([fidelity(k, s) for k, s in subs.items()])
fid.to_csv(OUT / f'subnet_fidelity_{HOST}.csv', index=False)
display(fid.round(3))

path L=3:   4,530 neurons,     496,997 connections
path L=4:  18,648 neurons,   3,039,649 connections
path L=5:  74,874 neurons,  12,335,607 connections
path L=6: 127,184 neurons,  20,187,912 connections
active set:  10,677 neurons,     926,228 connections


,network,neurons,connections,A 150Hz: DN corr,A 150Hz: DN mean ratio,A 150Hz: legMN firing,A 150Hz: readout spikes exact,B 75Hz: DN corr,B 75Hz: DN mean ratio,B 75Hz: legMN firing,B 75Hz: readout spikes exact,pass
0,path L=3,4530,496997,0.974,0.944,8 (full 9),0.029,0.962,0.987,8 (full 10),0.025,True
1,path L=4,18648,3039649,0.996,1.018,9 (full 9),0.046,0.997,0.980,9 (full 10),0.044,True
2,path L=5,74874,12335607,1.000,1.005,9 (full 9),0.056,1.000,1.009,9 (full 10),0.060,True
3,active set,10677,926228,1.000,0.998,9 (full 9),0.157,0.999,0.990,9 (full 10),0.159,True


## Step 4: benchmark

Wall-clock seconds of compute per 1 s of brain time, one trial at a time (as the closed loop runs), JO-C/E at 150 Hz, chosen `w_syn`, recording every spike. Median of 3 runs, after a warm-up (so compile / GPU start-up time is excluded; build time is reported separately).

Networks: the full network, every passing subnetwork, and the smallest path subnetwork for reference. Brian2 is timed too (CPU only); on Colab its full-network number comes from Step 1.

In [6]:
bench_nets = {'full': full}
bench_nets.update({n: subs[n] for n in fid[fid['pass']].network})
smallest = fid.sort_values('connections').network.iloc[0]
bench_nets.setdefault(smallest, subs[smallest])

rows = []
for name, sub in bench_nets.items():
    # Brian2 (CPU)
    if name in B2_TIME:
        rows.append(dict(network=name, engine='brian2', device='cpu', **B2_TIME[name]))
    elif name != 'full' or IN_COLAB:
        b2 = B2Fixed(sub, JO, verbose=False)
        b2.run(poisson_inputs(len(JO), RATE, 1.0, 2, seed=9), w_scale=W, verbose=False)
        rows.append(dict(network=name, engine='brian2', device='cpu', build_s=b2.build_s, s_per_sim_s=b2.run_s[-1]))
        del b2; gc.collect()
    for kind, (desc, needs_gpu) in ENGINES.items():
        if needs_gpu and not GPU:
            continue
        eng = make_engine(kind, sub, JO)
        t = time_engine(eng, rate=RATE, w_scale=W)
        rows.append(dict(network=name, engine=kind, device='gpu' if needs_gpu else 'cpu',
                         build_s=eng.build_s, s_per_sim_s=t))
        print(f"{name:12s} {kind:15s} {t:7.2f} s per simulated second")
        del eng; gc.collect()
        if GPU: torch.cuda.empty_cache()
bench = pd.DataFrame(rows).assign(host=HOST, cpu_cores=os.cpu_count(), gpu=gpu_name,
                                  neurons=lambda d: d.network.map({n: len(s['keep']) for n, s in bench_nets.items()}),
                                  connections=lambda d: d.network.map({n: len(s['w']) for n, s in bench_nets.items()}))
bench.to_csv(OUT / f'benchmark_{HOST}.csv', index=False)
display(bench.pivot_table(index=['network', 'neurons', 'connections'], columns='engine',
                          values='s_per_sim_s', sort=False).round(2))

full         numba              4.63 s per simulated second
full         torch-cpu         20.44 s per simulated second
path L=3     numba              0.19 s per simulated second
path L=3     torch-cpu          5.49 s per simulated second
path L=4     numba              0.55 s per simulated second
path L=4     torch-cpu          9.10 s per simulated second
path L=5     numba              1.91 s per simulated second
path L=5     torch-cpu         14.63 s per simulated second
active set   numba              0.51 s per simulated second
active set   torch-cpu          8.18 s per simulated second


,,engine,numba,torch-cpu,brian2
network,neurons,connections,,,
full,176422,24287791,4.63,20.44,NaN
path L=3,4530,496997,0.19,5.49,0.73
path L=4,18648,3039649,0.55,9.10,1.91
path L=5,74874,12335607,1.91,14.63,4.39
active set,10677,926228,0.51,8.18,1.63


**Throughput (sweeps, not closed loop):** several independent trials side by side on the GPU.

In [7]:
if GPU:
    eng = make_engine('torch-gpu', full, JO)
    for B in (1, 10, 32):
        print(f"full network, batch {B:>2}: {time_engine(eng, rate=RATE, w_scale=W, batch=B, reps=1):.2f} s per simulated second per trial")
    del eng; gc.collect(); torch.cuda.empty_cache()

## Step 5: decision

Rule (fixed in advance), for each machine measured:
1. Candidates: the full network, plus subnetworks that passed Step 3.
2. A candidate qualifies if its fastest engine on that machine needs **≤ 10 s per simulated second**.
3. Pick the **full network** if it qualifies (no fidelity loss, and the closed loop can add inputs freely). Otherwise pick the qualifying subnetwork with the best fidelity (highest worst-case DN correlation).

Run the notebook on both Colab and the laptop to fill in both rows.

In [8]:
results = {}
for f in sorted(OUT.glob('benchmark_*.csv')):
    host = f.stem.split('_', 1)[1]
    b = pd.read_csv(f)
    fd = pd.read_csv(OUT / f'subnet_fidelity_{host}.csv')
    passing = set(fd[fd['pass']].network) | {'full'}
    best = (b[b.engine != 'brian2'].sort_values('s_per_sim_s').groupby('network').first())
    best = best[best.index.isin(passing)]
    ok = best[best.s_per_sim_s <= 10]
    worst_corr = fd.set_index('network').filter(like='DN corr').min(axis=1)
    if 'full' in ok.index:
        pick = 'full'
    elif len(ok):
        pick = max(ok.index, key=lambda n: worst_corr.get(n, -1))
    else:
        pick = None
    results[host] = dict(network=pick,
                         engine=None if pick is None else best.loc[pick, 'engine'],
                         s_per_sim_s=None if pick is None else round(float(best.loc[pick, 's_per_sim_s']), 2),
                         full_best=f"{best.loc['full', 'engine']}: {best.loc['full', 's_per_sim_s']:.2f} s" if 'full' in best.index else None)
    print(f"{host:7s} -> {results[host]}")

dec = dict(w_scale=W, stimulus='JO-C/E', rule='full if <= 10 s per simulated second, else best-fidelity passing subnetwork',
           per_host=results)
(OUT / 'decision.json').write_text(json.dumps(dec, indent=2, default=str))
for host, r in results.items():
    if r['network'] not in (None, 'full') and r['network'] in subs:
        np.save(OUT / f"subnet_keep_{r['network'].replace(' ', '_').replace('=', '')}.npy", subs[r['network']]['keep'])
print(json.dumps(dec, indent=2, default=str))

laptop  -> {'network': 'full', 'engine': 'numba', 's_per_sim_s': 4.63, 'full_best': 'numba: 4.63 s'}
{
  "w_scale": 0.7,
  "stimulus": "JO-C/E",
  "rule": "full if <= 10 s per simulated second, else best-fidelity passing subnetwork",
  "per_host": {
    "laptop": {
      "network": "full",
      "engine": "numba",
      "s_per_sim_s": 4.63,
      "full_best": "numba: 4.63 s"
    }
  }
}
